# Initialize Git Repository
https://docs.ycrc.yale.edu/clusters-at-yale/guides/github/

In [ ]:
#git init -b main  # Initializes the repo with 'main' as the default branch
#git add .
#git commit -m "Initial commit"
#git remote add origin git@github.com:innacohen/mod-extract.git
#git push -u origin main --force

# Download mod files as a single JSON file

In [1]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# List to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None  # Default to None (indicating missing content)
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.RequestException as e:
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)  # Log failed download

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 467
n_rows = 20 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"], 
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")


/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Fetching .mod files:  20%|██        | 4/20 [00:00<00:00, 16.31it/s]

❌ Failed to fetch https://modeldb.science/getModelFile?model=58173&file=b05oct26/na.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/na.mod


Fetching .mod files: 100%|██████████| 20/20 [00:01<00:00, 15.58it/s]

⚠️ Some downloads failed. Check ../data/failed_downloads.json for details.

✅ All .mod files (including failures) saved in ../data/mod_files.json


# Read JSON file

In [8]:
pd.set_option("display.max_columns", None)

In [2]:
import os
import pandas as pd
import requests
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm

In [3]:
def View(df, rows=5, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`


In [29]:
# Function to extract mod directory from the URL
def extract_dir(url):
    match = re.search(r"file=([^/]+)/[^/]+\.mod", url)  # Extract the directory name before the .mod file
    return match.group(1) if match else None  # Return directory name if found, else None

# Function to extract mod file name without extension
def extract_fname(url):
    match = re.search(r"/([^/]+)\.mod$", url)  # Get filename without extension
    return match.group(1) if match else None  # Return only the name (e.g., 'na')

# Function to extract model_id from the URL
def extract_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

# Function to determine exclusion reason with snake_case labels
def determine_exclusion(row):
    if pd.isna(row["content"]):
        return "not_found"  # Standardized exclusion for missing files
    if "x86_64" in row["url"]:
        return "x86_64"  # Standardized exclusion for architecture issues
    return None  # Keep valid entries as None (not excluded)

# Function to extract all TITLE occurrences from .mod content
def extract_title(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"^TITLE\s+([^\n:]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None

# Function to extract all COMMENT sections from .mod content
def extract_comment(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"COMMENT\s+(.*?)(?:\s+ENDCOMMENT|\Z)", content, re.DOTALL)  
    return matches if matches else None

# Function to extract all SUFFIX occurrences from .mod content
def extract_suffix(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"SUFFIX\s+([^\n:\s]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None

# Function to extract all USEION occurrences from .mod content
def extract_use_ion(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"USEION\s+(\w+)", content)  
    return list(set(matches)) if matches else None  # Convert to set to remove duplicates


# Function to extract all ions listed after READ but stopping before WRITE, USEION, RANGE, or GLOBAL
def extract_read_ion(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    
    matches = re.findall(r"USEION\s+\w+\s+READ\s+([\w,\s]+?)(?=\s+(?:WRITE|USEION|RANGE|GLOBAL|NONSPECIFIC_CURRENT|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    # Flatten list, remove extra spaces, and split by commas/whitespace
    read_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return read_ions if read_ions else None  # Return list if found, else None


# Function to extract all ions listed after WRITE, stopping at the newline
def extract_write_ion(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"WRITE\s+([^\n:]+)", content, re.MULTILINE)  # Stop at comments
    
    if not matches:
        return None

    write_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return write_ions if write_ions else None  

# Function to extract all NONSPECIFIC currents
def extract_nonspecific_current(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"NONSPECIFIC_CURRENT\s+([^\n:]*)", content)

    if not matches:
        return None

    nonspecific_currents = [curr.strip() for match in matches for curr in re.split(r"[,\s]+", match) if curr]

    return nonspecific_currents if nonspecific_currents else None  

#todo: should we assume we only want active variables or also extract ones that are commented out?
# Function to extract RANGE variables based on mode
def extract_range(content, mode="active"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")


# Function to extract only active RANGE variables, stopping at colons and the end of the line
def extract_range(content):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract all RANGE statements (each line separately), stopping before colons
    matches = re.findall(r"^\s*RANGE\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    # Process active RANGE variables, ensuring they don't capture anything past the colon
    active_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return active_vars if active_vars else None  # Return only active variables
    
# Function to extract parameter names and values as a dictionary
def extract_parameter(content):
    if pd.isna(content):  
        return None
    
    # Capture parameter names and values, stopping at comments (`:`)
    matches = re.findall(r"(\w+)\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)(?=\s*(?:\n|:|$))", content)  
    
    param_dict = {name: float(value) for name, value in matches if re.match(r"^-?\d*\.?\d+(?:[eE][-+]?\d+)?$", value)}
    
    return param_dict if param_dict else None  


# Function to extract only active STATE variables, ignoring comments (`:`) and units `(mV)`, `(mA)`, etc.
def extract_state(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None

    # Find all STATE blocks and capture their contents
    matches = re.findall(r"STATE\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    state_vars = []
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore commented-out lines
                continue
            # Remove unit values in parentheses (e.g., `(mV)`, `(mA)`)
            clean_line = re.sub(r"\([^)]*\)", "", line).strip()
            if clean_line:  # Ensure non-empty lines
                state_vars.append(clean_line)

    return state_vars if state_vars else None  # Return cleaned list


# Function to extract only active GLOBAL variables, ignoring commented-out (`:`) ones
def extract_global(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None

    # Find all GLOBAL statements and capture their contents
    matches = re.findall(r"^\s*GLOBAL\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    # Process GLOBAL variables, ensuring they are properly split and cleaned
    global_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return global_vars if global_vars else None  # Return cleaned list

    
# Function to extract webpage heading
def extract_heading(url):
    try:
        response = requests.get(url, timeout=10)  # Fetch the webpage
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Try extracting heading from the most relevant tag
        heading = soup.find("h1")  # Look for <h1> (main title)
        return heading.text.strip() if heading else None  # Return text or None
    except requests.exceptions.RequestException:
        return None  # Return None if the request fail

# Function to extract citation (text inside parentheses)
def extract_citation(heading):
    if pd.isna(heading):
        return None
    match = re.search(r"\(([^)]+)\)", heading)  # Find text inside parentheses
    return match.group(1) if match else None  # Extract citation


# Function to extract first author(s) (removes "et al." and "al" correctly)
def extract_author(citation):
    if pd.isna(citation):
        return None

    # Extract first author(s) before "et al" or variants
    match = re.search(r"^([\w\s&\-,]+?)(?:\s+et\s+al\.?|et)?(?:,|\s|$)", citation)  
    first_author = match.group(1).strip() if match else None  

    # Remove any trailing "al" left behind
    if first_author:
        first_author = re.sub(r"\b(al)\b", "", first_author, flags=re.IGNORECASE).strip()

    return first_author

# Function to extract the first year from citation (including shortened years like '20)
def extract_year(citation):
    if pd.isna(citation):
        return None
    match = re.search(r"\b(19|20)?\d{2}\b|'\d{2}", citation)  # Find 4-digit or short year ('20)
    if match:
        return match.group(0).replace("'", "")  # Remove apostrophe but keep year as '20' if short
    return None  # Return None if no year found

In [18]:
# Load JSON into DataFrame
df = pd.read_json("../data/mod_files.json")

# Set "row_id" as the index
df.set_index("row_id", inplace=True)

# Exclude mod files
df["mod_exclude"] = df.apply(determine_exclusion, axis=1)  # Apply exclusion function

# Extract features from url
df["mod_heading"] = df["url"].apply(extract_heading)  # Extract heading from webpage
df["mod_citation"] = df["mod_heading"].apply(extract_citation)
df["mod_first_author"] = df["mod_citation"].apply(extract_author)
df["mod_year"] = df["mod_citation"].apply(extract_year)  # Now handles multiple years


df["mod_dir"] = df["url"].apply(extract_dir)
df["mod_name"] = df["url"].apply(extract_fname)
df["mod_model_id"] = df["url"].apply(extract_model_id)

#  Extract features from content
df["mod_title"] = df["content"].apply(extract_title)
df["mod_comment"] = df["content"].apply(extract_comment)
df["mod_suffix"] = df["content"].apply(extract_suffix)
df["mod_use_ion"] = df["content"].apply(extract_use_ion)
df["mod_read_ion"] = df["content"].apply(extract_read_ion)
df["mod_write_ion"] = df["content"].apply(extract_write_ion)
df["mod_nonspecific_current"] = df["content"].apply(extract_nonspecific_current)
df["mod_range"] = df["content"].apply(extract_range)
df["mod_global"] = df["content"].apply(extract_global)
df["mod_parameter"] = df["content"].apply(extract_parameter)
df["mod_state"] = df["content"].apply(extract_state)

In [19]:
#Questions
#Is it okay that we make assumptions like start after READ and stop after WRITE, what if there is no WRITE statement?
#How to capture variables that are commented out vs. not? 

"""example of a function that captures both active and commented out variables. I think its cleaner to capture active only?
import re
import pandas as pd

# Function to extract RANGE variables based on mode
def extract_range(content, mode="all"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")

"""

'example of a function that captures both active and commented out variables. I think its cleaner to capture active only?\nimport re\nimport pandas as pd\n\n# Function to extract RANGE variables based on mode\ndef extract_range(content, mode="all"):\n    if pd.isna(content):\n        return None  # Return None if content is missing\n\n    # Extract active RANGE variables (not commented out)\n    active_matches = re.findall(r"^\\s*RANGE\\s+([\\w\\s,]+)", content, re.MULTILINE)\n\n    # Extract commented-out RANGE variables (lines starting with ": RANGE")\n    commented_matches = re.findall(r"^\\s*:\\s*RANGE\\s+([\\w\\s,]+)", content, re.MULTILINE)\n\n    # Process active RANGE variables\n    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\\s]+", match) if var]\n\n    # Process commented-out RANGE variables\n    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\\s]+", match) if var]\n\n    if mode == "active":\n        r

In [37]:
df

,file_hash,raw_sha,count,url,download_url,content,mod_exclude,mod_heading,mod_citation,mod_first_author,mod_year,mod_dir,mod_name,mod_model_id,mod_title,mod_comment,mod_suffix,mod_use_ion,mod_read_ion,mod_write_ion,mod_nonspecific_current,mod_range,mod_global,mod_parameter,mod_state
row_id,,,,,,,,,,,,,,,,,,,,,,,,,
468,25b50b5441b9a5492122214e0648e0529ec055f2d6e97b...,f81e972ce9d730e924c8280e95395b26346853b9c0b397...,3,https://modeldb.science/58173?tab=2&file=b05oc...,https://modeldb.science/getModelFile?model=581...,None,not_found,None,None,None,None,b05oct26,na,58173,None,None,None,None,None,None,None,None,None,None,None
469,3fb039ac47d8004f4d9da9bc3da43bf89ce3e9c632e87d...,5a432e65a6d72ef8943f0ae6e833100108f341f793c0a0...,1,https://modeldb.science/116983?tab=2&file=thet...,https://modeldb.science/getModelFile?model=116...,TITLE I-h channel from Magee 1998 for distal d...,None,CA1 pyramidal neuron: h channel-dependent defi...,Marcelin et al. 2008,Marcelin,2008,theta,h,116983,[I-h channel from Magee 1998 for distal dendri...,None,[hd],None,None,None,[i],"[ghdbar, vhalfl, tau]","[linf, taua, taud]","{'kl': -12.0, 'q10': 4.5, 'qtl': 1.0}",[l]
470,338c3525e3a21e4cdc628b3224a949883cf12b2765f872...,e8daa7ec610c947cbfa1bef33e16ba2748e74da7e2596e...,2,https://modeldb.science/136026?tab=2&file=djur...,https://modeldb.science/getModelFile?model=136...,"\nCOMMENT\n\nna.mod\n\nSodium channel, Hodgkin...",None,Functional structure of mitral cell dendritic ...,Djurisic et al. 2008,Djurisic,2008,djurisic2008,na,136026,None,"[na.mod\n\nSodium channel, Hodgkin-Huxley styl...",[na],[na],[ena],[ina],None,"[m, h, gna, gbar, vshift]","[thm1, thm2, qm1, qm2, thi1, thi2, qi, qinf, t...",{'q10': 2.3},"[m, h]"
471,2ee9ed06c3f7f53d4c4a8f65c4f7ed7c979b810983fa72...,eeb928941300620229779ed4262c41055cac12f7b2fd02...,1,https://modeldb.science/18502?tab=2&file=spine...,https://modeldb.science/getModelFile?model=185...,\r\nCOMMENT\na synaptic current with NMDA func...,None,Spine fusion and branching affects synaptic re...,"Rusakov et al 1996, 1997",Rusakov,1996,spinebranches,nmdasyn,18502,None,[a synaptic current with NMDA function conduct...,None,None,None,None,[i],"[onset, tau1, tau2, Mgetha, gamma, gmax, e, i]",None,"{'Mgetha': 0.33, 'alpha': 0.0}",None
472,bcd979facda855ab577dc0cc18f1476bf510c7ac6b642e...,c5130c699809a9cc9fcd117e6257f56da565fdd1b282bc...,1,https://modeldb.science/256140?tab=2&file=Luqu...,https://modeldb.science/getModelFile?model=256...,TITLE Purkinje simplified Cell model\r\n\r\nCO...,None,Spike burst-pause dynamics of Purkinje cells r...,Luque et al 2019,Luque,2019,None,Purk_gM,256140,[Purkinje simplified Cell model\r],[kM channel\r\n \r\n Authors: Ni...,[Purk_gKM],[k],[ek],[ik],None,"[gkmbar, ik, g, M_inf, tau_M, M]",None,None,[M]
473,96526eb4a8f585ecbb64d59a643b0e83bfdc195f8bba6a...,bed60dff2ab3a09598fe1476f782be2bec39a0b686c866...,1,https://modeldb.science/82894?tab=2&file=nrntr...,https://modeldb.science/getModelFile?model=828...,COMMENT\nFour helpful hints:\n\n1) before call...,None,A single column thalamocortical network model ...,Traub et al 2005,Traub,2005,None,ri,82894,None,[Four helpful hints:\n\n1) before calling scal...,[nothing],None,None,None,None,None,None,{'x': 0.0},None
474,9bbabc95794782664eff3e6318db36ad8811c38572b48e...,63ca66c9d9f75f25001ff6a2ff622babc7851813152fb7...,1,https://modeldb.science/168858?tab=2&file=Cosk...,https://modeldb.science/getModelFile?model=168...,"TITLE Potasium dr type current for RD Traub, J...",None,Rhesus Monkey Layer 3 Pyramidal Neurons: Young...,Coskren et al. 2015,Coskren,2015,CoskrenEtAl2015,kdr,168858,"[Potasium dr type current for RD Traub, J Neur...",[Implemented by Aniruddha Yadav 2007 (aniruddh...,[kdr],[k],[ek],[ik],None,"[gbar, Vkd, ik, vrev, a1, a2, b1, b2]",None,None,[m]
475,20308b254cb6bb8a0e7f2db569dd2078b06b1f874dc1c7...,20308b254cb6bb8a0e7f2db569dd2078b06b1f874dc1c7...,1,https://modeldb.science/249705?tab=2&file=plat...,https://modeldb.science/getModelFile?model=249...,.

In [ ]:
df.columns

Flagging Issues

In [25]:
#row_id: 477 - commented out ranges included (https://modeldb.science/266818?tab=2&file=Ventricular_GUI.CircRes.ModelDB/Kss.mod)
#row_id: 481 - has comments with mod_state variables (https://modeldb.science/267511?tab=2&file=S1_Thal_NetPyNE_Frontiers_2022/sim/mod/ProbAMPANMDA_EMS.mod)
#row_id: 483 - has units in the mod_state ( https://modeldb.science/195666?tab=2&file=DewellGabbiani2018/mod_files/LGMD_KD_ca3.mod)


In [40]:
View(df.loc[:,["url","mod_range"]],50)

,url,mod_range
row_id,,
468,https://modeldb.science/58173?tab=2&file=b05oct26/na.mod,None
469,https://modeldb.science/116983?tab=2&file=theta/h.mod,"[ghdbar, vhalfl, tau]"
470,https://modeldb.science/136026?tab=2&file=djurisic2008/na.mod,"[m, h, gna, gbar, vshift]"
471,https://modeldb.science/18502?tab=2&file=spinebranches/nmdasyn.mod,"[onset, tau1, tau2, Mgetha, gamma, gmax, e, i]"
472,https://modeldb.science/256140?tab=2&file=LuqueEtAl2019/NEURON/Purk_gM.mod,"[gkmbar, ik, g, M_inf, tau_M, M]"
473,https://modeldb.science/82894?tab=2&file=nrntraub/mod/ri.mod,None
474,https://modeldb.science/168858?tab=2&file=CoskrenEtAl2015/kdr.mod,"[gbar, Vkd, ik, vrev, a1, a2, b1, b2]"
475,https://modeldb.science/249705?tab=2&file=plateau-potentials/mod/x86_64/glutamate.mod,None
476,https://modeldb.science/256024?tab=2&file=DewellGabbiani2019/mods/gap.mod,"[r, i]"


In [ ]:
! git add .
! git commit -m "fixed read ion"
! git push 

In [46]:
import re




In [ ]:
View(df,3)

In [44]:
sample = """ 
TITLE Ca2+ deinactivated K-D channel from RBD


UNITS {
	(mA) = (milliamp)
	(mV) = (millivolt)
	(S) = (siemens)
	(molar) = (1/liter)
	(mM) = (millimolar)
}

NEURON {
    THREADSAFE
    : note - every variable accessible in NEURON will be having the suffix _KD

        SUFFIX KD_ca3
        USEION k READ ek WRITE ik
        USEION ca READ cai
        RANGE gmax, g, taun, taul
        GLOBAL tnmax, tlmax
}

PARAMETER {
	: all values can be adjusted in hoc files
	gmax=0.01 	(mho/cm2)
	vhalfn=-49	(mV)
	vn2=-60		(mV)
	zn=7.0		(mV)
	tnmax=10	(ms)
	tnmin=1.5	(ms)
	tns=-5.5	(mV)
	np=2
	
	vhalfl=-70	(mV)
	zl=-3.5		(mV)
	tlmax=1050	(ms)
	tlmin=70	(ms)
	vl2=-75		(mV)
	tls=15		(mV)
	
	cavm=18		(mV)
	lcp=1		(1)
	kD=4e-4		(mM)
	tauca=20	(ms)
}

STATE {
	n
	l
	ov	(mV)
	ovs	(mV)
}

ASSIGNED {
    v (mV)
    ek (mV)
    cai (mM)

	ik (mA/cm2)
	ninf
	linf
	vs	(mV)
	taul (ms)
	taun (ms)
	g (S/cm2)
}

BREAKPOINT {
	SOLVE states METHOD cnexp
	l = 1/(1+exp((vhalfl-ov+ovs)/zl))
	g = gmax*n^np*l
	ik = g*(v-ek)
}

INITIAL {
	lf(v)
	rates(v)
	n = ninf
	l = 1/(1+exp((vhalfl-v)/zl)*exp(vs/zl))
	ov = v
	ovs = vs
}

FUNCTION alpn(v(mV)) {
  alpn = exp((vhalfn-v)/zn)
}

FUNCTION betn(v(mV)) {
  betn = exp((vn2-v)/tns) 
}

DERIVATIVE states {
	lci(cai)
	rates(v)
	n' = (ninf - n)/taun
	:l' = (linf - l)/taul
	:l' = (1/zl)*exp((vhalfl-)/zl)*exp(vs/zl)/(1+exp((vhalfl-v)/zl)*exp(vs/zl))^2
	:l' = linf - l
	ov' = (v-ov)/taul
	ovs' = (vs-ovs)/tauca
}

PROCEDURE lf(v(mV)) {
	LOCAL dvdt, dsdt
	lci(cai)
	:linf = 1/(1+exp((vhalfl-v)/zl)*exp(vs/zl))
	linf = 1/(1+exp((vhalfl-ov+ovs)/zl))
}

PROCEDURE rates(v (mV)) { :callable from hoc
	LOCAL a
	TABLE ninf, taun, taul DEPEND vhalfn, tlmax, tnmax, tnmin
          FROM -100 TO 50 WITH 600
    
	a = alpn(v)
	ninf = 1/(1 + a)
	taun = 4*(tnmax-tnmin)/(1+betn(v))*ninf+tnmin
	taul = 2*tlmax/( exp((v-vl2)/tls) + exp((vl2-v)/tls) ) + tlmin
}


PROCEDURE lci(cai (mM)) { :callable from hoc
	TABLE vs DEPEND lcp, kD, cavm
          FROM 0 TO 0.02 WITH 500
    
	vs = cavm-cavm/(1+(cai/kD)^lcp)
    
"""

In [47]:
extract_parameter(sample)

{'gmax': 0.01,
 'vhalfn': -49.0,
 'vn2': -60.0,
 'zn': 7.0,
 'tnmax': 10.0,
 'tnmin': 1.5,
 'tns': -5.5,
 'np': 2.0,
 'vhalfl': -70.0,
 'zl': -3.5,
 'tlmax': 1050.0,
 'tlmin': 70.0,
 'vl2': -75.0,
 'tls': 15.0,
 'cavm': 18.0,
 'lcp': 1.0,
 'kD': 0.0004,
 'tauca': 20.0}